# 08: The live pipeline, step by step

Notebooks 01-07 built and validated everything on historical matches. This one uses the `chimera` package, which does the same thing for a match that hasn't happened yet, and explains the pick with live pre-match news (online RAG).

Run top to bottom. Paths resolve from the project root, so no `os.chdir` is needed.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import chimera  # sets the OpenMP flag before torch/lightgbm load
from chimera.engine import ChimeraEngine
from chimera.features import MatchSpec, PlayerSpec
from chimera.rag import RagOptions
from datetime import date

engine = ChimeraEngine()  # loads history + both models once
print('history through', engine.store.last_history_date.date())

## 1. Resolving names

Cricsheet stores names as initials + surname. The resolver maps normal names onto them, using the team to break ties (there are several R. Sharmas).

In [ ]:
for name, team in [('Virat Kohli', 'RCB'), ('Varun Chakravarthy', 'KKR'), ('Rohit Sharma', 'MI'), ('Some Debutant', 'MI')]:
    t, _ = engine.store.canonical_team(team)
    r = engine.store.resolve_player(name, t)
    print(f'{name:20s} -> {str(r.name):12s} ({r.method}, confidence {r.confidence})')

## 2. Live features for one player

Each feature is computed from the player's matches *before* the match date, the same thing `shift(1)` did during training. Compare these numbers with the columns you built in notebook 03.

In [ ]:
spec = MatchSpec(team1='RCB', team2='KKR', venue='Chinnaswamy', match_date=date(2026, 4, 10), players=[
    PlayerSpec(n, 'RCB') for n in ['Virat Kohli', 'Phil Salt', 'Rajat Patidar', 'Liam Livingstone', 'Jitesh Sharma',
                                   'Tim David', 'Krunal Pandya', 'Bhuvneshwar Kumar', 'Josh Hazlewood', 'Yash Dayal', 'Suyash Sharma']
] + [
    PlayerSpec(n, 'KKR') for n in ['Quinton de Kock', 'Sunil Narine', 'Ajinkya Rahane', 'Venkatesh Iyer', 'Rinku Singh',
                                   'Andre Russell', 'Ramandeep Singh', 'Harshit Rana', 'Varun Chakravarthy', 'Vaibhav Arora', 'Spencer Johnson']
])

built = engine.builder.build(spec)
print(built.match['venue'], '|', built.match['weather_source'])
built.features.set_index('player').loc['V Kohli', ['matches_played', 'rolling_avg_fantasy_5', 'rolling_avg_fantasy_10',
    'venue_avg_fantasy', 'opposition_avg_fantasy', 'batting_position', 'batting_position_source', 'is_home', 'weather_dew']]

## 3. Predict

The toss is unknown here, so each player is predicted with `won_toss=1` and `won_toss=0` and the two are averaged.

In [ ]:
pred = engine.predict(spec)
for w in pred.warnings:
    print('!', w)
pred.public_players()[['player', 'team', 'role', 'batting_position', 'lgbm_pred', 'lstm_pred', 'ensemble_pred']].head(10)

## 4. Optimize

Same integer program as notebook 06, now with include/exclude support.

In [ ]:
from chimera.optimizer import TeamConstraints

sel = engine.optimize(pred.players, TeamConstraints(include=['V Kohli']))
print(f'{sel.credits_used:.1f} credits, {sel.projected_points:.1f} projected points')
sel.team[['player', 'team', 'role', 'credit_value', 'ensemble_pred', 'captain', 'vice_captain', 'final_points']]

## 5. Explain with live news (online RAG)

Searches Google News (and GNews if you set a key) for the week before the match, chunks and embeds what it finds, retrieves the most relevant pieces, and asks the LLM to explain the XI citing them.

You can also pass your own article URLs or pasted text through `RagOptions`.

In [ ]:
expl = engine.explain(pred.match, sel.team, RagOptions(urls=[], manual_texts=[]))
print(f'provider={expl.provider} model={expl.model} retrieval={expl.retrieval_backend} docs={expl.documents_found}')
for n in expl.source_notes: print(' ', n)
print()
print(expl.text)
for c in expl.citations:
    print(f"[{c['id']}] {c['title']} ({c['publisher']}) {c['url']}")
for w in expl.warnings: print('!', w)